# LDaCA Wordflow: Notebook Launcher

This notebook starts LDaCA Wordflow inside a Jupyter/Binder environment.

The code cell below will:
1. Start the backend + frontend on a single port (no nginx required).
2. Display a clickable link to open the web app.

> **Data root:** The backend defaults to `DATA_ROOT=~/Documents/ldaca`. Override by setting the `DATA_ROOT` environment variable before running.

In [ ]:
from ldaca_wordflow import start_async_server
from utils import configure_hub_networking, display_app_link

PORT = 8887
# The v0.7 backend only answers allowlisted Host/Origin values. Detect the
# hub host the browser really uses (one throwaway fetch through the proxy)
# and export TRUSTED_HOSTS / CORS_ALLOWED_ORIGINS before settings load.
await configure_hub_networking()
# start_async_server() runs Uvicorn as a task on this notebook's event loop
# and returns a ServerHandle only after startup completes — no polling needed.
handle = await start_async_server(port=PORT)
await display_app_link(port=PORT)

### Inspecting the stored data

- Since v0.7, workspace state is owned by the running server (the old module-level `workspace_manager` no longer exists), so this cell inspects what the server has persisted on disk instead
- User files and imports live under `DATA_ROOT/users/<user>/`; workspace snapshots live under `DATA_ROOT/workspaces/`
- If this is your first run, the folders are created automatically when the backend starts

In [ ]:
# The ServerHandle from the launcher cell carries the resolved settings.
data_root = handle.settings.get_data_root()
print(f"Data root: {data_root}\n")

workspaces = [
    p for p in sorted((data_root / "workspaces").glob("*")) if not p.name.startswith(".")
]
if not workspaces:
    print("No workspaces exist yet — create one in the app.")
else:
    print("Workspaces:")
    for w in workspaces:
        print(f"  - {w.name}")

users_root = data_root / "users"
if users_root.is_dir():
    print("\nUsers:", ", ".join(p.name for p in sorted(users_root.iterdir()) if p.is_dir()))